In [8]:
import nltk
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to
[nltk_data]    |     C:\Users\CSE\AppData\Roaming\nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     C:\Users\CSE\AppData\Roaming\nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     C:\Users\CSE\AppData\Roaming\nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     C:\Users\CSE\AppData\Roaming\nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     C:\Users\CSE\AppData\Roaming\nltk_data...
[nltk_data]

True

In [16]:
import nltk
from sklearn.model_selection import train_test_split
from nltk.corpus import twitter_samples, stopwords
import pandas as pd
from sklearn.linear_model import LogisticRegression
import re
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

stop_words = set(stopwords.words('english'))

# 간단한 텍스트 전처리 함수 -------------------------------------------
def preprocess_text(text):
    # 1) 소문자
    text = text.lower()
    # 2) 숫자 제거 (원하면 유지해도 됨)
    text = re.sub(r'\d+', ' ', text)
    # 3) 특수문자/구두점 제거
    text = re.sub(r'[^a-z\s]', ' ', text)
    # 4) 불용어 제거
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    # 5) 다시 공백으로 결합
    return ' '.join(tokens)


#  공통 학습/평가 함수 --------------------------------------------------
def train_and_evaluate(vectorizer, X_train, X_test, y_train, y_test, name="Model"):
    print(f"\n========== {name} ==========")
    # 1) 텍스트 -> 벡터화
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    print("특징 수(feature dimension):", X_train_vec.shape[1])

    # 2) 분류 모델 (로지스틱 회귀)
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train_vec, y_train)

    # 3) 예측
    y_pred = clf.predict(X_test_vec)

    # 4) 평가
    acc = accuracy_score(y_test, y_pred)
    print("Accuracy:", acc)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['ham', 'spam']))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    return clf, vectorizer

# 새 문장에 대해 스팸 예측 ---------------------------------------------
def predict_spam(texts, model, vectorizer):
    # texts: 문자열 리스트
    cleaned = [preprocess_text(t) for t in texts]
    X_vec = vectorizer.transform(cleaned)
    preds = model.predict(X_vec)
    for t, p in zip(texts, preds):
        label = "SPAM" if p == 1 else "HAM"
        print(f"[{label}] {t}")



In [ ]:

pos_tweets = twitter_samples.strings('positive_tweets.json')
neg_tweets = twitter_samples.strings('negative_tweets.json')

print("긍정 트윗 수:", len(pos_tweets))
print("부정 트윗 수:", len(neg_tweets))

# pos/neg 합치기
docs = pos_tweets + neg_tweets
labels = [1] * len(pos_tweets) + [0] * len(neg_tweets) 
#labels = ['pos'] * len(pos_tweets) + ['neg'] * len(neg_tweets)

print("문서 개수:", len(docs))   # 10,000개


# 전처리 적용
clean_docs = [preprocess_text(doc) for doc in docs]

print("\n=== 전처리 전/후 예시 ===")
for i in range(3):
    print(f"원문 {i}: ", docs[i][:200], "...")
    print(f"전처리 {i}: ", clean_docs[i][:200], "...")
    print("-" * 60)

sentences = [doc.split() for doc in clean_docs]

긍정 트윗 수: 5000
부정 트윗 수: 5000
문서 개수: 10000

=== 전처리 전/후 예시 ===
원문 0:  #FollowFriday @France_Inte @PKuchly57 @Milipol_Paris for being top engaged members in my community this week :) ...
전처리 0:  followfriday france inte pkuchly milipol paris top engaged members community week ...
------------------------------------------------------------
원문 1:  @Lamb2ja Hey James! How odd :/ Please call our Contact Centre on 02392441234 and we will be able to assist you :) Many thanks! ...
전처리 1:  lamb ja hey james odd please call contact centre able assist many thanks ...
------------------------------------------------------------
원문 2:  @DespiteOfficial we had a listen last night :) As You Bleed is an amazing track. When are you in Scotland?! ...
전처리 2:  despiteofficial listen last night bleed amazing track scotland ...
------------------------------------------------------------


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    clean_docs,
    labels, 
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("\n훈련 문서 수:", len(X_train))
print("테스트 문서 수:", len(X_test))



훈련 문서 수: 8000
테스트 문서 수: 2000


In [18]:
# 6. TF-IDF 벡터화
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=5,
) 

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("벡터차원", X_train_vec.shape[1])

벡터차원 1735


In [20]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train)

# 8. 평가
y_pred = clf.predict(X_test_vec)

print("\n=== 평가 결과 ===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['neg', 'pos']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# 9. 새 트윗 감정 예측 함수
def predict_sentiment(texts):
    """
    texts: 문자열 리스트
    """
    clean = [preprocess_text(t) for t in texts]
    vec   = vectorizer.transform(clean)
    preds = clf.predict(vec)

    for t, p in zip(texts, preds):
        label = "긍정" if p == "pos" else "부정"
        print(f"[{label}] {t}")

# 10. 예시 문장 테스트
print("\n=== 새 문장 예측 예시 ===")
sample_texts = [
    "I love this new phone, it is amazing!",
    "This is the worst service I have ever experienced.",
    "Not bad, but could be better.",
    "I am so happy today :)"
]
predict_sentiment(sample_texts)


=== 평가 결과 ===
Accuracy : 0.7515

Classification Report:
              precision    recall  f1-score   support

         neg       0.74      0.79      0.76      1000
         pos       0.77      0.72      0.74      1000

    accuracy                           0.75      2000
   macro avg       0.75      0.75      0.75      2000
weighted avg       0.75      0.75      0.75      2000


Confusion Matrix:
[[785 215]
 [282 718]]

=== 새 문장 예측 예시 ===
[긍정] I love this new phone, it is amazing!
[부정] This is the worst service I have ever experienced.
[부정] Not bad, but could be better.
[긍정] I am so happy today :)
